In [2]:
# Cell 1 — Imports + paths

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import sys

project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

from src.job_intel.config import PROCESSED_DATA_DIR, CH2_PROCESSED_DF
from src.job_intel.pipelines.chapter2_hidden_structures import run_chapter2_hidden_structures
from src.job_intel.features.skill_specialisation_map import compute_skill_specialisation_lift
from src.job_intel.models.skill_prob_matrix import build_skill_probability_matrix


In [3]:
# Cell 2 — Helper assertions (small, readable checks)

def assert_true(condition: bool, msg: str) -> None:
    if not condition:
        raise AssertionError(msg)

def assert_close(a: float, b: float, tol: float, msg: str) -> None:
    if not np.isfinite(a) or not np.isfinite(b):
        raise AssertionError(f"{msg} (non-finite values)")
    if abs(a - b) > tol:
        raise AssertionError(f"{msg} (|{a} - {b}| > {tol})")


In [4]:
# Cell 3 — Load the Chapter 2 base dataframe (the contract input)
from src.job_intel.features.skills_pca import SKILL_COLS

base = pd.read_csv(CH2_PROCESSED_DF)
mask = base[SKILL_COLS].sum(axis=1) > 0
base = base.loc[mask].copy()



assert_true("job_id" in base.columns or base.index.name == "job_id",
            "CH2_PROCESSED_DF must contain job_id (column or index).")

# Standardise to job_id column for downstream joins
if "job_id" not in base.columns and base.index.name == "job_id":
    base = base.reset_index()

assert_true(base["job_id"].is_unique, "base job_id must be unique (1 row per job).")

print("OK: base loaded")
print("  rows:", len(base))
print("  cols:", base.shape[1])


OK: base loaded
  rows: 6110
  cols: 139


In [5]:
# Cell 4 — CHAPTER 2 SANITY CHECK (pipeline artefacts)

# Run chapter 2 pipeline without overwriting anything
km_jobs_df, undirected_edges = run_chapter2_hidden_structures(
    threshold=0.5,
    save_graph_pickle=False,
)

# ---- job families checks ----
assert_true(isinstance(km_jobs_df, pd.DataFrame), "km_jobs_df must be a DataFrame.")
assert_true(set(km_jobs_df.columns) >= {"job_id", "job_family_id"},
            "km_jobs_df must contain ['job_id','job_family_id'].")

assert_true(km_jobs_df["job_id"].is_unique, "km_jobs_df job_id must be unique.")
assert_true(km_jobs_df["job_family_id"].notna().all(), "job_family_id must have no NAs.")
assert_true(km_jobs_df["job_family_id"].nunique() >= 2, "Need >=2 clusters to be meaningful.")

# Joinability check
j = base[["job_id"]].merge(km_jobs_df[["job_id", "job_family_id"]], on="job_id", how="inner")
assert_true(len(j) == len(base),
            "Job families do not cover all jobs in base (inner join dropped rows).")

# ---- skill similarity edges checks ----
assert_true(isinstance(undirected_edges, pd.DataFrame), "undirected_edges must be a DataFrame.")
assert_true(set(undirected_edges.columns) >= {"skill_1", "skill_2", "similarity"},
            "undirected_edges must contain ['skill_1','skill_2','similarity'].")

assert_true((undirected_edges["skill_1"] != undirected_edges["skill_2"]).all(),
            "Skill similarity edges must not include self-edges.")
assert_true(undirected_edges[["skill_1", "skill_2"]].notna().all().all(),
            "Skill similarity edges must not have missing endpoints.")
assert_true(undirected_edges["similarity"].notna().all(),
            "Skill similarity edges must not have missing similarity values.")

# Similarity range check (cosine-ish dot product of L2-normalised vectors)
assert_true((undirected_edges["similarity"] <= 1.000001).all(),
            "Similarity values exceed 1 (unexpected for normalised dot products).")
assert_true((undirected_edges["similarity"] >= -1.000001).all(),
            "Similarity values below -1 (unexpected for normalised dot products).")

# Dedup check: unique undirected pairs
pair_dupes = undirected_edges.duplicated(subset=["skill_1", "skill_2"]).sum()
assert_true(pair_dupes == 0, f"Duplicate undirected pairs found: {pair_dupes}")

print("OK: Chapter 2 pipeline sanity checks passed")
print("  km_jobs_df:", km_jobs_df.shape, "| clusters:", km_jobs_df["job_family_id"].nunique())
print("  undirected_edges:", undirected_edges.shape)


=== Chapter 2: Build job–skill bipartite graph ===
✅ Building the skill probability matrix...
✅ Skill probability matrix built.
✅ Creating graph...
✅ Adding nodes...
✅ Adding edges using threshold = 0.5.
✅ Edges added. Total edges = 40306
✅ Graph successfully built!
Graph ready. Nodes=6137, Edges=40306, threshold=0.5
=== Chapter 2: Train Node2Vec + extract embeddings ===


Computing transition probabilities:   0%|          | 0/6137 [00:00<?, ?it/s]

Generating walks (CPU: 4): 100%|██████████| 10/10 [01:24<00:00,  8.42s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Embeddings extracted: job_emb=(6110, 64), skill_emb=(27, 64)
=== Chapter 2: Cluster job embeddings into job families ===
Job embeddings injected. Shape=(6110, 64)
Embeddings normalised using 'l2' normalisation.
Checking data and normalisation...
  NAs in raw job embeddings: 0
  NAs in normalised embeddings: 0
  Shape maintained? True
  Unit norm check (first 10 rows): [1.         1.         0.9999999  1.         1.         0.9999998
 1.         0.99999994 1.0000001  0.9999999 ]
KMeans fitted. k=20, random_state=42.
Output rows=6110 (should equal n_jobs=6110).
=== Chapter 2: Build skill similarity edge list from skill embeddings ===
Normalising skill embeddings...
Sanity checks for normalised embeddings...
  NAs in raw skill embeddings: 0
  NAs in normalised embeddings: 0
  Shape maintained? True
  Unit norm check (first 10 rows): [0.9999999  1.0000001  1.         0.99999994 1.         1.
 1.         1.         1.         0.9999999 ]
Computing skill similarity matrix...
Sanity checks fo

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from src.job_intel.config import PROCESSED_DATA_DIR

# -----------------------------
# Inputs expected in notebook:
# - undirected_edges: DataFrame from current pipeline run
#   columns: ["skill_1","skill_2","similarity"] (or equivalent)
# - km_jobs_df: DataFrame from current pipeline run
#   columns: ["job_id","job_family_id"]
# -----------------------------

# -----------------------------
# Load baselines (saved earlier)
# -----------------------------
BASELINE_SKILL_EDGES = PROCESSED_DATA_DIR / "skill_similarity_edges_k5_embeddings.csv"
BASELINE_JOB_FAMILIES = PROCESSED_DATA_DIR / "job_families_graph_embeddings.csv"

edges_ref = pd.read_csv(BASELINE_SKILL_EDGES)
jobs_ref = pd.read_csv(BASELINE_JOB_FAMILIES)

# If baseline job families has an accidental unnamed column, drop it safely
for c in ["Unnamed: 0"]:
    if c in jobs_ref.columns:
        jobs_ref = jobs_ref.drop(columns=[c])

# -----------------------------
# Helper: canonicalise edge pairs
# -----------------------------
def canonicalise_edges(df: pd.DataFrame, a="skill_1", b="skill_2") -> pd.DataFrame:
    df = df.copy()
    if a not in df.columns or b not in df.columns:
        raise ValueError(f"Expected columns '{a}', '{b}' in edge df. Got: {df.columns.tolist()}")
    df["skill_1"] = df[[a, b]].min(axis=1)
    df["skill_2"] = df[[a, b]].max(axis=1)
    if "similarity" not in df.columns:
        raise ValueError("Expected 'similarity' column in edge df.")
    return df[["skill_1", "skill_2", "similarity"]]

edges_now = canonicalise_edges(undirected_edges)
edges_ref = canonicalise_edges(edges_ref)

# -----------------------------
# 1) Skill neighbour stability
# -----------------------------
def neighbours_from_undirected(edges: pd.DataFrame, k: int = 5) -> dict[str, set[str]]:
    """
    Build top-k neighbour sets per skill from an undirected edge list.
    """
    # Make symmetric adjacency
    a = edges.rename(columns={"skill_1": "src", "skill_2": "dst"})[["src", "dst", "similarity"]]
    b = edges.rename(columns={"skill_2": "src", "skill_1": "dst"})[["src", "dst", "similarity"]]
    adj = pd.concat([a, b], ignore_index=True)

    neigh = {}
    for src, g in adj.groupby("src"):
        topk = g.sort_values("similarity", ascending=False).head(k)
        neigh[src] = set(topk["dst"].tolist())
    return neigh

k = 5
neigh_now = neighbours_from_undirected(edges_now, k=k)
neigh_ref = neighbours_from_undirected(edges_ref, k=k)

skills_common = sorted(set(neigh_now.keys()) & set(neigh_ref.keys()))
if len(skills_common) == 0:
    raise AssertionError("No overlapping skill nodes between current and baseline neighbour sets.")

def jaccard(a: set, b: set) -> float:
    if len(a | b) == 0:
        return 1.0
    return len(a & b) / len(a | b)

jac_scores = []
for s in skills_common:
    jac_scores.append(jaccard(neigh_now[s], neigh_ref[s]))

jac_scores = np.array(jac_scores)

print("Skill neighbour stability (per-skill Jaccard, top-k)")
print(f"  skills compared: {len(skills_common)}")
print(f"  mean Jaccard:    {jac_scores.mean():.3f}")
print(f"  median Jaccard:  {np.median(jac_scores):.3f}")
print(f"  min Jaccard:     {jac_scores.min():.3f}")

# Thresholds: tune if needed, but these are reasonable for stochastic embeddings
assert np.median(jac_scores) >= 0.60, "Skill neighbour stability too low (median Jaccard < 0.60)."

# -----------------------------
# 2) Job family stability (label-permutation invariant)
# -----------------------------
if not {"job_id", "job_family_id"}.issubset(set(km_jobs_df.columns)):
    raise ValueError(f"km_jobs_df must have columns ['job_id','job_family_id']. Got: {km_jobs_df.columns.tolist()}")
if not {"job_id", "job_family_id"}.issubset(set(jobs_ref.columns)):
    raise ValueError(f"jobs_ref must have columns ['job_id','job_family_id']. Got: {jobs_ref.columns.tolist()}")

m_now = km_jobs_df[["job_id", "job_family_id"]].copy()
m_ref = jobs_ref[["job_id", "job_family_id"]].copy()

merged = m_now.merge(m_ref, on="job_id", how="inner", suffixes=("_now", "_ref"))
if len(merged) == 0:
    raise AssertionError("No overlapping job_ids between current and baseline job-family mappings.")

ari = adjusted_rand_score(merged["job_family_id_ref"], merged["job_family_id_now"])

print("\nJob family clustering stability (ARI)")
print(f"  jobs compared: {len(merged)}")
print(f"  ARI:           {ari:.4f}")

# ARI >= 0.65 is "reasonably stable" for stochastic graph embeddings.
assert ari >= 0.65, f"Job family clustering drifted too much (ARI={ari:.4f})."

print("\n✅ Cell 5 stability checks passed (meaningful thresholds).")


Skill neighbour stability (per-skill Jaccard, top-k)
  skills compared: 27
  mean Jaccard:    0.668
  median Jaccard:  0.667
  min Jaccard:     0.111

Job family clustering stability (ARI)
  jobs compared: 6110
  ARI:           0.5800


AssertionError: Job family clustering drifted too much (ARI=0.5800).

In [ ]:
# Cell 6 — SKILL SPECIALISATION MAP sanity test (job_family_id)
# Uses: base + skill probability matrix + job families

# Build skill probability matrix directly from the base (this matches your Chapter 2 contract)
# IMPORTANT: this assumes build_skill_probability_matrix returns probs with columns like 'prob_*' (27 cols)
prob_mat = build_skill_probability_matrix(jobs_df=base.set_index("job_id") if "job_id" in base.columns else base)

# Standardise prob_mat for compute_skill_specialisation_lift (expects job_id col)
if prob_mat.index.name == "job_id":
    prob_mat = prob_mat.reset_index()

# job families already computed above (km_jobs_df)

lift_job_family = compute_skill_specialisation_lift(
    base_df=base,
    skill_prob_df=prob_mat,
    job_families=km_jobs_df,
    group_col="title_rich",
    expected_n_skills=27,
    min_group_n=None,
    verbose=True,
    show_plots=False,
    save_plots=False,
    save_data=False,
)

assert_true(isinstance(lift_job_family, pd.DataFrame), "Lift output must be a DataFrame.")
assert_true(lift_job_family.shape[1] == 27, "Lift must have 27 skill columns.")
assert_true(lift_job_family.shape[0] >= 2, "Lift must have >=2 groups (rows).")
assert_true(np.isfinite(lift_job_family.to_numpy()).all(), "Lift contains non-finite values.")

# Weighted mean lift across groups should be ~0 for each skill
group_sizes = base.merge(km_jobs_df, on="job_id", how="inner")["title_rich"].value_counts()
group_sizes = group_sizes.reindex(lift_job_family.index)
weights = (group_sizes / group_sizes.sum()).to_numpy()

weighted_mean_lift = (lift_job_family.to_numpy().T @ weights)  # vector length=27
max_abs = float(np.max(np.abs(weighted_mean_lift)))

assert_true(max_abs < 1e-10,
            f"Weighted mean lift not ~0 (max_abs={max_abs:.3e}). Likely a join/alignment issue.")

print("OK: Skill specialisation lift sanity checks passed (title_rich)")
print("  lift shape:", lift_job_family.shape)


In [ ]:
# Cell 7 — SKILL SPECIALISATION MAP smoke test on an existing categorical column (e.g., Sector)
# (Change group_col as needed: 'Sector', 'job_title_family', 'ownership_clean', 'Size', 'state')

group_col = "Sector"
assert_true(group_col in base.columns, f"base is missing '{group_col}'")

lift_sector = compute_skill_specialisation_lift(
    base_df=base,
    skill_prob_df=prob_mat,
    job_families=None,      # not needed
    group_col=group_col,
    expected_n_skills=27,
    min_group_n=10,         # optional stabiliser
    verbose=True,
    show_plots=False,
    save_plots=False,
    save_data=False,
)

assert_true(lift_sector.shape[1] == 27, "Lift must have 27 skill columns.")
assert_true(lift_sector.shape[0] >= 2, "Lift must have >=2 groups (rows).")

print("OK: Skill specialisation lift smoke test passed:", group_col)
print("  lift shape:", lift_sector.shape)
